# BPMN Assistant — QLoRA SFT + DPO (Kaggle GPU)

Fine-tunes a self-hostable open model (**Qwen3-8B**, Apache-2.0) on the BPMN datasets, in two stages per the TDD:

1. **SFT** (QLoRA) on `data/instruction/*` (capabilities C1–C4, C6, C7)
2. **DPO** on `data/preference/*` (preference alignment)

Produces small **LoRA adapters** you download and commit to the project (`models/`).

## Before you run
1. **Add your data**: run `python src/training/prepare_data.py` locally, then upload `data/training/` (the 4 `*.jsonl` files) as a **Kaggle Dataset**. Attach it and set `DATA_DIR` below (e.g. `/kaggle/input/bpmn-training-data`).
2. **Enable GPU**: Settings → Accelerator → **GPU T4 x2** (or P100).
3. **Enable Internet**: Settings → Internet **On** (to download the base model).
4. Run all cells top to bottom. ≈ 1–3 h on a T4.

## Correctness notes (verified against current TRL, 2025–2026)
- Sequence length is **`SFTConfig(max_length=...)`** — NOT `max_seq_length` (removed in modern TRL; its 1024 default would silently truncate long BPMN XML). We set `MAX_LEN=2048` and print a token-length report so you can confirm nothing is truncated.
- Tokenizer is passed as **`processing_class=`** (TRL ≥ 0.12; `tokenizer=` was removed).
- **Qwen3 thinking mode**: the default chat template injects `<think>` tags. For plain BPMN XML / answers we render with **`enable_thinking=False`** so no `<think>` leaks into targets.
- QLoRA + gradient checkpointing needs **`use_reentrant=False`** + **`enable_input_require_grads()`** or you get a no-grad error.

> TRL's API moves fast; cell 1 prints installed versions. If an arg errors, check the version and adjust.

In [ ]:
# 1) Dependencies (pin to a recent TRL so max_length / processing_class / assistant_only_loss exist)
!pip install -q -U "transformers>=4.51" "trl>=0.20" "peft>=0.13" "datasets>=2.20" "bitsandbytes>=0.44" "accelerate>=1.0"
import torch, transformers, trl, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| trl", trl.__version__, "| peft", peft.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# (optional) Authenticate to the HF Hub for faster, rate-limit-free downloads.
# Add your token on Kaggle: Add-ons -> Secrets -> new secret named HF_TOKEN. Safe to skip: Qwen3 is public.
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded from Kaggle Secrets.")
except Exception:
    print("No HF_TOKEN secret set - continuing unauthenticated (fine for the public Qwen3 model).")

In [ ]:
# 2) Config
BASE_MODEL   = "Qwen/Qwen3-8B"        # Apache-2.0, self-hostable (TDD). Fallback: "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR   = "/kaggle/working"
MAX_LEN      = 3072                  # SFTConfig.max_length / DPOConfig.max_length. MUST exceed the longest
                                     # BPMN XML target (verified by the length report in cell 3). Lower to
                                     # 1536 only if the report shows ~0 examples over the limit AND you OOM.
ENABLE_THINKING = False              # Qwen3: keep thinking OFF -> no <think> tags in BPMN targets/prompts
SFT_EPOCHS   = 2
DPO_EPOCHS   = 1

SYSTEM_PROMPT = (
    "You are a BPMN 2.0 expert assistant. Answer precisely and follow BPMN 2.0 conventions. "
    "When asked to generate a diagram, output valid BPMN 2.0 XML. When asked to review a "
    "diagram, identify concrete issues and how to fix them."
)

# Auto-detect the attached dataset folder. Kaggle mounts a dataset at /kaggle/input/<slug>, where
# <slug> comes from the dataset URL (NOT its display name) and files may be nested -- so we locate
# the folder that actually contains sft_train.jsonl instead of hardcoding a path that may not match.
import os, glob
_hits = glob.glob("/kaggle/input/**/sft_train.jsonl", recursive=True)
assert _hits, (
    "sft_train.jsonl not found under /kaggle/input. Present: "
    + str(os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'no /kaggle/input')
    + ". Attach your dataset via 'Add Input', and ensure it has the 4 *.jsonl files."
)
DATA_DIR = os.path.dirname(_hits[0])
print("Using DATA_DIR =", DATA_DIR)
print("files:", sorted(os.listdir(DATA_DIR)))

In [ ]:
# 3) Tokenizer + SFT dataset (render chat -> text, thinking OFF) + LENGTH FILTER
import re, numpy as np
from transformers import AutoTokenizer
from datasets import load_dataset

tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

sft = load_dataset("json", data_files={"train": f"{DATA_DIR}/sft_train.jsonl", "val": f"{DATA_DIR}/sft_val.jsonl"})

def render(ex):
    text = tok.apply_chat_template(ex["messages"], tokenize=False,
                                   add_generation_prompt=False, enable_thinking=ENABLE_THINKING)
    # Qwen3's template injects an EMPTY <think></think> block even with enable_thinking=False
    # (known multi-turn quirk). Strip empty think blocks so they never enter the training targets.
    text = re.sub(r"<think>\s*</think>\s*", "", text)
    return {"text": text}

sft = sft.map(render, remove_columns=sft["train"].column_names)

def _ntok(t): return len(tok(t, add_special_tokens=False)["input_ids"])

lens = [_ntok(t) for t in sft["train"]["text"]]
over = sum(1 for L in lens if L > MAX_LEN)
print("token lengths  p50=%d  p95=%d  p99=%d  max=%d" % (np.percentile(lens,50), np.percentile(lens,95), np.percentile(lens,99), max(lens)))
print("examples over MAX_LEN=%d: %d (%.1f%%) -> DROPPED, not truncated" % (MAX_LEN, over, 100*over/len(lens)))

# Drop over-length examples rather than silently truncating: a cut-off BPMN XML target teaches the
# model to emit invalid, incomplete XML. Raise MAX_LEN (cell 2) to keep more; lower it if you OOM.
for _split in list(sft.keys()):
    _before = len(sft[_split])
    sft[_split] = sft[_split].filter(lambda ex: _ntok(ex["text"]) <= MAX_LEN)
    print("%s: kept %d/%d" % (_split, len(sft[_split]), _before))

assert len(sft["train"]) > 0, "all examples dropped - raise MAX_LEN"
assert "<think>" not in sft["train"][0]["text"], "thinking tags leaked - check enable_thinking"
print("--- sample (first 600 chars) ---")
print(sft["train"][0]["text"][:600])

In [ ]:
# 4) Load base model in 4-bit (QLoRA) + attach LoRA
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4/P100 have no bf16 -> fp16
    bnb_4bit_use_double_quant=True)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map="auto", trust_remote_code=True)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.enable_input_require_grads()   # REQUIRED: grad flow through checkpointing + frozen 4-bit base

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

In [ ]:
# 5) Stage 1 — SFT (QLoRA). max_length (NOT max_seq_length); use_reentrant=False for checkpointing.
from trl import SFTConfig, SFTTrainer

sft_cfg = SFTConfig(
    output_dir=f"{OUTPUT_DIR}/sft",
    per_device_train_batch_size=1, gradient_accumulation_steps=16,
    num_train_epochs=SFT_EPOCHS, learning_rate=2e-4, warmup_ratio=0.03,
    lr_scheduler_type="cosine", fp16=True, logging_steps=20,
    save_strategy="epoch", eval_strategy="epoch",
    max_length=MAX_LEN, dataset_text_field="text",
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit", report_to="none")

sft_trainer = SFTTrainer(
    model=model, args=sft_cfg,
    train_dataset=sft["train"], eval_dataset=sft["val"],
    peft_config=lora, processing_class=tok)
sft_trainer.train()
sft_trainer.save_model(f"{OUTPUT_DIR}/bpmn-sft-adapter")
tok.save_pretrained(f"{OUTPUT_DIR}/bpmn-sft-adapter")
print("SFT adapter saved.")

# OPTIONAL quality boost: instead of the rendered 'text' + full-sequence loss above, pass the raw
# conversational dataset (with 'messages') and set assistant_only_loss=True to train on completions
# only. It improves instruction-following BUT ensure MAX_LEN exceeds every example (see cell 3) —
# with truncation it can silently zero the loss (TRL issue #3927). Verify train loss > 0.

In [ ]:
# 6) Stage 2 — DPO on top of the SFT adapter
import torch, gc
del model, sft_trainer; gc.collect(); torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from trl import DPOConfig, DPOTrainer
from datasets import load_dataset

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                            device_map="auto", trust_remote_code=True)
base.config.use_cache = False
# resume the SFT adapter as the trainable policy; ref model = adapter-disabled frozen base (PEFT default,
# so ref_model=None is correct). Reference is the BASE model — standard QLoRA-DPO recipe.
model = PeftModel.from_pretrained(base, f"{OUTPUT_DIR}/bpmn-sft-adapter", is_trainable=True)
model.enable_input_require_grads()

dpo = load_dataset("json", data_files={
    "train": f"{DATA_DIR}/dpo_train.jsonl", "val": f"{DATA_DIR}/dpo_val.jsonl"})

def fmt(ex):
    prompt = tok.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": ex["prompt"]}],
        tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING)
    return {"prompt": prompt, "chosen": ex["chosen"], "rejected": ex["rejected"]}

dpo = dpo.map(fmt, remove_columns=[c for c in dpo["train"].column_names if c not in ("prompt", "chosen", "rejected")])

dpo_cfg = DPOConfig(
    output_dir=f"{OUTPUT_DIR}/dpo",
    per_device_train_batch_size=1, gradient_accumulation_steps=16,
    num_train_epochs=DPO_EPOCHS, learning_rate=5e-6, beta=0.1,
    fp16=True, logging_steps=10, save_strategy="epoch",
    max_prompt_length=1024, max_length=MAX_LEN,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit", report_to="none")

dpo_trainer = DPOTrainer(model=model, ref_model=None, args=dpo_cfg,
                         train_dataset=dpo["train"], eval_dataset=dpo["val"], processing_class=tok)
dpo_trainer.train()
dpo_trainer.save_model(f"{OUTPUT_DIR}/bpmn-dpo-adapter")
tok.save_pretrained(f"{OUTPUT_DIR}/bpmn-dpo-adapter")
print("DPO adapter saved.")

In [ ]:
# 7) Quick sanity generations with the fine-tuned model (thinking OFF)
def chat(msg, max_new_tokens=500):
    text = tok.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": msg}],
        tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING)
    ids = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

print(chat("What is the difference between an Error Boundary Event and an Escalation Boundary Event?"))
print("\n---\n")
print(chat("Generate a BPMN 2.0 diagram: an employee submits a timesheet, a manager approves or rejects it, and the employee is notified."))

In [ ]:
# 8) Package adapters for download
import shutil
for name in ["bpmn-sft-adapter", "bpmn-dpo-adapter"]:
    shutil.make_archive(f"{OUTPUT_DIR}/{name}", "zip", f"{OUTPUT_DIR}/{name}")
    print("zipped:", f"{OUTPUT_DIR}/{name}.zip")
print("\nDownload the .zip files from the Kaggle 'Output' panel (right sidebar) after the run.")

## After the run — save into the project
1. Download `bpmn-dpo-adapter.zip` (final model) from Kaggle's **Output** panel.
2. Unzip into the repo under `models/bpmn-dpo-adapter/` (git-ignored).
3. Evaluate with the harness (needs a GPU box, or a second Kaggle notebook):
   ```bash
   python src/eval/run_eval.py --runner hf:Qwen/Qwen3-8B   # baseline
   # then load base + adapter and score the fine-tuned model to measure lift
   ```
4. For serving, keep base+adapter, or merge (`PeftModel.merge_and_unload()`) → full weights (~16 GB) for a vLLM/Ollama endpoint (the on-prem path).

**Tips:** OOM → lower `MAX_LEN` to 1536 (only if the cell-3 report shows ~0 truncated) or switch `BASE_MODEL` to `Qwen/Qwen2.5-3B-Instruct`. Flash-attention is **not** available on T4 (SM75) — do not enable it. To use both T4s, launch with `accelerate`.